In [2]:
# 1. Install necessary libraries
!pip install transformers datasets torch torchvision tqdm

import sys
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tqdm import tqdm

# Project root (when running from notebooks/ or from repo root)
ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Image transform (must match HeritageDataset; used for training and generation)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# --- 1. ENCODER (ResNet50) ---
class HeritageEncoder(nn.Module):
    def __init__(self):
        super(HeritageEncoder, self).__init__()
        resnet = models.resnet50(weights='ResNet50_Weights.DEFAULT')
        # Remove the last two layers to get spatial feature maps (7x7x2048)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])

        # FREEZE ENCODER: As per project plan, we only train the decoder/attention
        for param in self.resnet.parameters():
            param.requires_grad = False

    def forward(self, images):
        features = self.resnet(images)  # [batch, 2048, 7, 7]
        features = features.permute(0, 2, 3, 1)  # [batch, 7, 7, 2048]
        features = features.view(features.size(0), -1, features.size(-1)) # [batch, 49, 2048]
        return features

# --- 2. ATTENTION (Bahdanau) ---
class HeritageAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(HeritageAttention, self).__init__()
        self.W = nn.Linear(decoder_dim, attention_dim)
        self.U = nn.Linear(encoder_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1)

    def forward(self, features, hidden):
        # features: [batch, 49, 2048], hidden: [batch, 768]
        hidden_with_time = hidden.unsqueeze(1)
        score = torch.tanh(self.W(hidden_with_time) + self.U(features))
        weights = torch.softmax(self.v(score), dim=1)
        context = torch.sum(weights * features, dim=1)
        return context, weights

# --- 3. THE COMPLETE SYSTEM ---
class HeritageLens(nn.Module):
    def __init__(self, vocab_size):
        super(HeritageLens, self).__init__()
        self.encoder = HeritageEncoder()
        self.attention = HeritageAttention(encoder_dim=2048, decoder_dim=768, attention_dim=512)
        self.gpt2 = GPT2LMHeadModel.from_pretrained('gpt2')

        # Bridge to map visual context to GPT2 hidden size
        self.visual_projection = nn.Linear(2048, 768)

    def forward(self, images, captions, labels=None):
        # captions: input_ids [batch, seq_len]. labels: same shape, -100 for padding (optional)
        # Get visual features: [batch, 49, 2048]
        features = self.encoder(images)

        # For training, we use global average pooling for the initial visual context
        visual_context = self.visual_projection(features.mean(dim=1))

        # Get GPT2 word embeddings
        inputs_embeds = self.gpt2.transformer.wte(captions)

        # Inject visual context into the word embeddings
        combined_embeds = inputs_embeds + visual_context.unsqueeze(1)

        # Pass through GPT2 (use labels for loss so padding can be masked with -100)
        loss_labels = labels if labels is not None else captions
        outputs = self.gpt2(inputs_embeds=combined_embeds, labels=loss_labels)
        return outputs.loss, outputs.logits

In [ ]:
# --- DATA: HeritageDataset + tokenization ---
from src.data.heritage_dataset import HeritageDataset

DATA_ROOT = ROOT / "data"
JSON_PATH = DATA_ROOT / "processed" / "metadata.json"
IMAGES_DIR = DATA_ROOT / "raw" / "wikimedia" / "images"

dataset = HeritageDataset(json_path=JSON_PATH, images_dir=IMAGES_DIR, transform=transform)

# Collate: batch (image, caption_str) -> (images, input_ids, attention_mask, labels)
MAX_LENGTH = 64

def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    captions = [b[1] for b in batch]
    encoded = tokenizer(
        captions,
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return images, input_ids, attention_mask, labels

train_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

# Initialize the model and move to GPU
model = HeritageLens(vocab_size=len(tokenizer)).to(device)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# --- TRAINING LOOP ---
NUM_EPOCHS = 3

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for images, input_ids, attention_mask, labels in pbar:
        images = images.to(device)
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        loss, logits = model(images, input_ids, labels=labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")

# Save checkpoint (optional)
ckpt_dir = ROOT / "outputs" / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), ckpt_dir / "heritage_lens.pt")
print(f"Checkpoint saved to {ckpt_dir / 'heritage_lens.pt'}")
print("Training complete.")

In [ ]:
def generate_caption(image_path, model, tokenizer, max_length=30):
    """Generate a caption for an image. Uses transform from cell 0."""
    model.eval()

    # 1. Load and preprocess the image (transform defined in cell 0)
    from PIL import Image
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    # 2. Start with the "start" token
    input_ids = torch.tensor([[tokenizer.bos_token_id]]).to(device)

    generated_caption = []

    with torch.no_grad():
        for _ in range(max_length):
            # Get the model's prediction
            loss, logits = model(image, input_ids)

            # Take the last word predicted
            next_token_logits = logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

            # If the model predicts "End of Text", stop generating
            if next_token_id == tokenizer.eos_token_id:
                break

            generated_caption.append(next_token_id.item())

            # Append predicted word to input for the next step
            input_ids = torch.cat([input_ids, next_token_id], dim=-1)

    # Convert IDs back to actual Nepali/English words
    return tokenizer.decode(generated_caption, skip_special_tokens=True)

# --- TEST IT OUT ---
# You can upload an image to Colab's sidebar and put the path here
# print(generate_caption("path_to_your_temple_image.jpg", model, tokenizer))